# 02 Labeling Dataset — выбранные category-runs

Берёт FAISS embedding candidates из `01_candidate_generation.ipynb` и делает стратифицированный CSV для ручной разметки. По умолчанию работает с одним `DEDUP_CATEGORY_RUN`; для fine-tuning reranker можно собрать один большой датасет по нескольким категориям.

## Инструкция по ручной разметке

Заполняйте колонку `label` одним из значений: `exact_duplicate`, `different_product`, `uncertain`. В `notes` можно кратко объяснить сомнение или правило.

- `exact_duplicate`: тот же базовый товар для модели, даже если отличается фасовка или multipack. Пример: `Соус соевой, 500 мл` vs `Соус соевой, 500 мл - 2 шт`, если бренд/тип/вкус совпадают.
- `different_product`: похожая карточка того же brand/веса, но другой вкус/тип продукта. Пример hard-negative из EDA/candidates: `Соус Барбекю "Ноль грамм", 330г для мяса...` vs `Соус Сладкий чили "Ноль грамм" 330г...`, тот же вес, но разные flavor-токены.
- `uncertain`: данных недостаточно или пара спорная без просмотра карточки/состава. Пример: `Горчица Баварская HAAS 1 кг.` vs `Горчица Haas Дижонская, 3 шт по 170 г` — похожий brand, но одновременно меняются тип и pack.

`labeling_stratum`, `is_cross_marketplace_pair`, `is_hard_negative_candidate` и `is_pack_variant_candidate` — подсказки для отбора, а не правильный ответ. Score-страты `high_similarity`, `medium_similarity` и `random_easy_negative` по умолчанию считаются относительно текущего распределения FAISS cosine внутри каждой категории, потому что абсолютный масштаб зависит от embedding-модели и категории. Pack-variant пары теперь размечаются как `exact_duplicate`, если это тот же базовый товар; конкретная фасовка будет выделяться позже правилами.

Для большого датасета под дообучение reranker используйте `DEDUP_LABELING_CATEGORY_RUNS=sauces,coconut_oil,soap` и `DEDUP_LABELING_TARGET_SIZE=3000`: notebook поделит размер примерно поровну между категориями, а внутри каждой сохранит страты `cross_marketplace`, `hard_negative`, `pack_variant`, `high/medium/low similarity`.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    LabelingSamplingConfig,
    add_cross_marketplace_flags,
    add_pack_variant_flags,
    labeling_score_strata_masks,
    split_labeling_target_size,
    stratified_labeling_sample,
    resolve_category_run,
    resolve_run_paths,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 140)


In [ ]:
def parse_category_runs() -> list:
    raw_runs = os.environ.get("DEDUP_LABELING_CATEGORY_RUNS", "").strip()
    if raw_runs:
        requested_runs = [part.strip() for part in raw_runs.replace(";", ",").split(",") if part.strip()]
    else:
        requested_runs = [os.environ.get("DEDUP_CATEGORY_RUN", "").strip() or None]

    runs = []
    seen_slugs = set()
    for requested_run in requested_runs:
        run = resolve_category_run(requested_run)
        if run.slug in seen_slugs:
            continue
        runs.append(run)
        seen_slugs.add(run.slug)
    return runs


CATEGORY_RUNS = parse_category_runs()
RUN_PATHS_BY_SLUG = {run.slug: resolve_run_paths(PROJECT_ROOT, run) for run in CATEGORY_RUNS}
IS_MULTI_RUN = len(CATEGORY_RUNS) > 1


def candidates_path_for_run(run) -> Path:
    override = os.environ.get("DEDUP_CANDIDATES_PATH")
    if override and not IS_MULTI_RUN:
        return Path(override).expanduser()
    return RUN_PATHS_BY_SLUG[run.slug].candidates_path


def default_labeling_path() -> Path:
    if not IS_MULTI_RUN:
        return RUN_PATHS_BY_SLUG[CATEGORY_RUNS[0].slug].labeling_path
    suffix = "_".join(run.artifact_suffix for run in CATEGORY_RUNS)
    return PROJECT_ROOT / "research" / "dedup" / "data" / f"labeling_{suffix}.csv"


LABELING_PATH = Path(os.environ.get("DEDUP_LABELING_PATH", default_labeling_path())).expanduser()
TARGET_SAMPLE_SIZE = int(os.environ.get("DEDUP_LABELING_TARGET_SIZE", "3000" if IS_MULTI_RUN else "400"))
RANDOM_STATE = int(os.environ.get("DEDUP_LABELING_RANDOM_STATE", "42"))
SCORE_STRATIFICATION = os.environ.get("DEDUP_LABELING_SCORE_STRATIFICATION", "quantile").strip() or "quantile"
HIGH_SIMILARITY_TOP_SHARE = float(os.environ.get("DEDUP_LABELING_HIGH_TOP_SHARE", "0.25"))
EASY_NEGATIVE_BOTTOM_SHARE = float(os.environ.get("DEDUP_LABELING_EASY_BOTTOM_SHARE", "0.25"))
HIGH_SIMILARITY_THRESHOLD = float(os.environ.get("DEDUP_LABELING_HIGH_THRESHOLD", "0.72"))
MEDIUM_SIMILARITY_LOWER = float(os.environ.get("DEDUP_LABELING_MEDIUM_LOWER", "0.50"))
MEDIUM_SIMILARITY_UPPER = float(os.environ.get("DEDUP_LABELING_MEDIUM_UPPER", "0.72"))
EASY_NEGATIVE_UPPER = float(os.environ.get("DEDUP_LABELING_EASY_UPPER", "0.50"))

def make_sampling_config(target_size: int, *, run_offset: int = 0) -> LabelingSamplingConfig:
    return LabelingSamplingConfig(
        target_size=target_size,
        random_state=RANDOM_STATE + run_offset * 1000,
        score_stratification=SCORE_STRATIFICATION,
        high_similarity_top_share=HIGH_SIMILARITY_TOP_SHARE,
        easy_negative_bottom_share=EASY_NEGATIVE_BOTTOM_SHARE,
        high_similarity_threshold=HIGH_SIMILARITY_THRESHOLD,
        medium_similarity_lower=MEDIUM_SIMILARITY_LOWER,
        medium_similarity_upper=MEDIUM_SIMILARITY_UPPER,
        easy_negative_upper=EASY_NEGATIVE_UPPER,
    )


RUN_TARGET_SIZES = split_labeling_target_size(TARGET_SAMPLE_SIZE, len(CATEGORY_RUNS))
RUN_SAMPLING_CONFIGS = {
    run.slug: make_sampling_config(target_size, run_offset=idx)
    for idx, (run, target_size) in enumerate(zip(CATEGORY_RUNS, RUN_TARGET_SIZES, strict=True))
}
sampling_config = make_sampling_config(TARGET_SAMPLE_SIZE)

print(
    "Category runs: "
    + ", ".join(f"{run.slug} — {run.display_name}; project: {run.project_name}" for run in CATEGORY_RUNS)
)
print(f"Labeling path: {LABELING_PATH}")
print(f"Target sample size: {sampling_config.target_size}")
print(f"Random state: {sampling_config.random_state}")
print(f"Score stratification: {sampling_config.score_stratification}")
if sampling_config.score_stratification == "quantile":
    print(
        "Score buckets: "
        f"top {sampling_config.high_similarity_top_share:.0%} -> high_similarity; "
        f"bottom {sampling_config.easy_negative_bottom_share:.0%} -> random_easy_negative"
    )
else:
    print(
        "Absolute score thresholds: "
        f"high >= {sampling_config.high_similarity_threshold}; "
        f"medium [{sampling_config.medium_similarity_lower}, {sampling_config.medium_similarity_upper}); "
        f"easy < {sampling_config.easy_negative_upper}"
    )

display(
    pd.DataFrame(
        [
            {
                "category_run": run.slug,
                "category_name": run.display_name,
                "project_name": run.project_name,
                "candidate_path": str(candidates_path_for_run(run)),
                "sample_target": RUN_SAMPLING_CONFIGS[run.slug].target_size,
                "random_state": RUN_SAMPLING_CONFIGS[run.slug].random_state,
            }
            for run in CATEGORY_RUNS
        ]
    )
)


In [ ]:
candidate_frames = {}
load_rows = []

for run in CATEGORY_RUNS:
    candidates_path = candidates_path_for_run(run)
    if not candidates_path.exists():
        raise FileNotFoundError(
            f"Не найден {candidates_path}. Сначала выполните notebooks/01_candidate_generation.ipynb "
            f"для category-run {run.slug}."
        )

    frame = pd.read_csv(candidates_path)
    frame = add_cross_marketplace_flags(add_pack_variant_flags(frame))
    frame["category_run"] = run.slug
    frame["category_name"] = run.display_name
    frame["project_name"] = run.project_name
    candidate_frames[run.slug] = frame
    load_rows.append(
        {
            "category_run": run.slug,
            "candidate_pairs": len(frame),
            "sample_target": RUN_SAMPLING_CONFIGS[run.slug].target_size,
            "candidate_path": str(candidates_path),
        }
    )

load_summary = pd.DataFrame(load_rows)
print(f"Loaded candidates: {load_summary['candidate_pairs'].sum():,} pairs across {len(candidate_frames)} category-run(s)")
display(load_summary)
display(candidate_frames[CATEGORY_RUNS[0].slug].head(5))


In [ ]:
availability_rows = []
score_bucket_ranges = []

for run in CATEGORY_RUNS:
    candidates = candidate_frames[run.slug]
    run_config = RUN_SAMPLING_CONFIGS[run.slug]
    score = pd.to_numeric(candidates["baseline_similarity_score"], errors="coerce").fillna(0.0)
    score_masks = labeling_score_strata_masks(candidates, run_config)
    availability_rows.extend(
        [
            {
                "category_run": run.slug,
                "stratum": "cross_marketplace_candidate",
                "available_pairs": int(candidates["is_cross_marketplace_pair"].fillna(False).sum()),
            },
            {
                "category_run": run.slug,
                "stratum": "hard_negative_candidate",
                "available_pairs": int(candidates["is_hard_negative_candidate"].fillna(False).sum()),
            },
            {
                "category_run": run.slug,
                "stratum": "pack_variant_candidate",
                "available_pairs": int(candidates["is_pack_variant_candidate"].fillna(False).sum()),
            },
            {
                "category_run": run.slug,
                "stratum": "high_similarity",
                "available_pairs": int(score_masks["high_similarity"].sum()),
            },
            {
                "category_run": run.slug,
                "stratum": "medium_similarity",
                "available_pairs": int(score_masks["medium_similarity"].sum()),
            },
            {
                "category_run": run.slug,
                "stratum": "random_easy_negative",
                "available_pairs": int(score_masks["random_easy_negative"].sum()),
            },
        ]
    )

    for stratum in ["high_similarity", "medium_similarity", "random_easy_negative"]:
        values = score[score_masks[stratum]]
        score_bucket_ranges.append(
            {
                "category_run": run.slug,
                "stratum": stratum,
                "pairs": int(len(values)),
                "min_score": values.min() if len(values) else pd.NA,
                "median_score": values.median() if len(values) else pd.NA,
                "max_score": values.max() if len(values) else pd.NA,
            }
        )

availability = pd.DataFrame(availability_rows)
display(availability)
display(pd.DataFrame(score_bucket_ranges))


In [ ]:
labeling_frames = []
for run in CATEGORY_RUNS:
    run_labeling = stratified_labeling_sample(candidate_frames[run.slug], RUN_SAMPLING_CONFIGS[run.slug])
    run_labeling["category_run"] = run.slug
    run_labeling["category_name"] = run.display_name
    run_labeling["project_name"] = run.project_name
    labeling_frames.append(run_labeling)

labeling_df = pd.concat(labeling_frames, ignore_index=True) if labeling_frames else pd.DataFrame()
labeling_df = labeling_df.sort_values(
    ["category_run", "labeling_stratum", "baseline_similarity_score", "sku_a", "sku_b"],
    ascending=[True, True, False, True, True],
).reset_index(drop=True)

export_columns = [
    "label",
    "notes",
    "category_run",
    "category_name",
    "project_name",
    "labeling_stratum",
    "raw_record_id_a",
    "raw_record_id_b",
    "marketplace_a",
    "marketplace_b",
    "marketplaces_a",
    "marketplaces_b",
    "sku_a",
    "sku_b",
    "title_a",
    "title_b",
    "brand_a",
    "brand_b",
    "subcategory_a",
    "subcategory_b",
    "subcategory_relation",
    "unit_amount_a",
    "unit_amount_b",
    "total_amount_a",
    "total_amount_b",
    "multipack_count_a",
    "multipack_count_b",
    "embedding_similarity_score",
    "candidate_rank",
    "candidate_source",
    "blocking_scope",
    "baseline_similarity_score",
    "is_cross_marketplace_pair",
    "is_hard_negative_candidate",
    "is_pack_variant_candidate",
]
labeling_df = labeling_df[[column for column in export_columns if column in labeling_df.columns]].copy()

LABELING_PATH.parent.mkdir(parents=True, exist_ok=True)
labeling_df.to_csv(LABELING_PATH, index=False)
print(f"Saved labeling dataset: {LABELING_PATH}")
print(f"Rows saved: {len(labeling_df):,}")


In [ ]:
category_stats = (
    labeling_df.groupby(["category_run", "category_name", "project_name"])
    .agg(
        pairs=("category_run", "size"),
        cross_marketplace_pairs=("is_cross_marketplace_pair", "sum"),
        hard_negative_pairs=("is_hard_negative_candidate", "sum"),
        pack_variant_pairs=("is_pack_variant_candidate", "sum"),
    )
    .reset_index()
    .sort_values("category_run")
)
display(category_stats)

strata_stats = (
    labeling_df.groupby(["category_run", "labeling_stratum"])
    .agg(
        pairs=("labeling_stratum", "size"),
        cross_marketplace_pairs=("is_cross_marketplace_pair", "sum"),
    )
    .reset_index()
    .sort_values(["category_run", "labeling_stratum"])
)
display(strata_stats)

score_by_stratum = labeling_df.groupby(["category_run", "labeling_stratum"])["baseline_similarity_score"].agg(
    pairs="count",
    min="min",
    median="median",
    max="max",
).reset_index()
display(score_by_stratum)

cross_marketplace_summary = (
    labeling_df.assign(
        same_marketplace_pair=~labeling_df["is_cross_marketplace_pair"].fillna(False)
    )
    .groupby("category_run")
    .agg(
        rows_total=("category_run", "size"),
        cross_marketplace_pairs=("is_cross_marketplace_pair", "sum"),
        same_marketplace_pairs=("same_marketplace_pair", "sum"),
    )
    .reset_index()
)
display(cross_marketplace_summary)


## Sanity-check before manual work

Проверяем, что экспорт не содержит автоматически заполненных labels и что каждая страта выглядит как ожидаемый тип ручного контроля.


In [ ]:
assert labeling_df["label"].fillna("").eq("").all(), "label должен оставаться пустым для ручной разметки"
assert labeling_df["notes"].fillna("").eq("").all(), "notes должен оставаться пустым для ручной разметки"
assert len(labeling_df) <= sampling_config.target_size

for row in strata_stats[["category_run", "labeling_stratum"]].itertuples(index=False):
    print(f"{row.category_run} / {row.labeling_stratum}")
    display(
        labeling_df[
            labeling_df["category_run"].eq(row.category_run)
            & labeling_df["labeling_stratum"].eq(row.labeling_stratum)
        ]
        .sort_values("baseline_similarity_score", ascending=False)
        .head(3)
    )


## Итог

Главный артефакт этого ноутбука — CSV в `LABELING_PATH`: для одиночного run это `labeling_<suffix>.csv`, для multi-run — общий файл вроде `labeling_sauces_coconut_oil_soap.csv`. После ручной разметки этот файл можно использовать как gold-set для сравнения matching engines и как обучающий датасет для fine-tuning reranker.
